# Contact Map Visualization

Compare model-generated vs ground-truth residue-residue contact maps as heatmaps.
Downloads a PDB structure, computes ground-truth atomic contacts, then generates
contacts from a trained exp4 checkpoint with varying prefix sizes.

In [ ]:
# Config
CHECKPOINT_PATH = "../../outputs/exp4"  # auto-finds latest checkpoint-* subdir
PDB_ID = "1QYS"
CONTACT_DISTANCE_CUTOFF = 4.0  # Angstroms, for heavy atom contacts
MAX_NEW_TOKENS = 3440
PREFIX_SIZES = [0, 5, 10, 20]
N_ROLLOUTS = 10  # number of sampled rollouts per prefix size
DEVICE = "cuda"

In [15]:
# Download & parse PDB structure
import tempfile
import numpy as np
from biotite.database import rcsb
from biotite.structure.io import pdbx
from biotite.structure import filter_amino_acids

path = rcsb.fetch(PDB_ID, "cif", tempfile.gettempdir())
cif = pdbx.CIFFile.read(path)
atoms = pdbx.get_structure(cif.block, model=1)

# First chain, amino acids only, heavy atoms only
first_chain = atoms.chain_id[0]
chain_atoms = atoms[(atoms.chain_id == first_chain) & filter_amino_acids(atoms) & (atoms.element != "H")]

# Build residue list (1-indexed positions)
res_ids = chain_atoms.res_id
unique_res = sorted(set(res_ids))
res_id_to_pos = {rid: i + 1 for i, rid in enumerate(unique_res)}  # 1-indexed
sequence = [chain_atoms[chain_atoms.res_id == rid].res_name[0] for rid in unique_res]
seq_len = len(sequence)
print(f"Protein {PDB_ID}: {seq_len} residues, chain {first_chain}")
print(f"Sequence: {' '.join(sequence[:10])}...")

Protein 1QYS: 92 residues, chain A
Sequence: ASP ILE GLN VAL GLN VAL ASN ILE ASP ASP...


In [16]:
# Compute ground-truth contacts
from scipy.spatial import KDTree
from experiments.exp4_contact_prediction.src.data import VALID_ATOMS

coords = chain_atoms.coord
atom_names = chain_atoms.atom_name
atom_res_ids = chain_atoms.res_id

# Filter to atoms known in the vocabulary
known_atom_set = set(VALID_ATOMS.get(sequence[0], set()))  # just for reference
all_known_atoms = set()
for aa in VALID_ATOMS:
    all_known_atoms.update(VALID_ATOMS[aa])

# Use KDTree for efficient neighbor search
tree = KDTree(coords)
pairs = tree.query_pairs(r=CONTACT_DISTANCE_CUTOFF)

gt_contacts = []  # (pos1, pos2, atom1, atom2)
for i, j in pairs:
    ri, rj = atom_res_ids[i], atom_res_ids[j]
    pi, pj = res_id_to_pos.get(ri), res_id_to_pos.get(rj)
    if pi is None or pj is None:
        continue
    if abs(pi - pj) < 2:
        continue
    ai, aj = str(atom_names[i]), str(atom_names[j])
    if ai not in all_known_atoms or aj not in all_known_atoms:
        continue
    # Validate atoms for their respective amino acids
    aa_i = sequence[pi - 1]
    aa_j = sequence[pj - 1]
    if aa_i not in VALID_ATOMS or ai not in VALID_ATOMS[aa_i]:
        continue
    if aa_j not in VALID_ATOMS or aj not in VALID_ATOMS[aa_j]:
        continue
    # Canonical ordering: larger position first for the pair with larger separation
    if pi > pj:
        gt_contacts.append((pj, pi, aj, ai))
    else:
        gt_contacts.append((pi, pj, ai, aj))

# Deduplicate
gt_contacts = sorted(set(gt_contacts), key=lambda c: (-abs(c[0] - c[1]), c[0], c[1], c[2], c[3]))
print(f"Ground-truth contacts: {len(gt_contacts)}")
print(f"First 5: {gt_contacts[:5]}")

# Build the document prompt
seq_tokens = " ".join(f"<{aa}>" for aa in sequence)
base_prompt = f"<deterministic-positives-only> <begin_sequence> {seq_tokens} <begin_contacts>"
print(f"\nPrompt length: {len(base_prompt.split())} tokens")

Ground-truth contacts: 860
First 5: [(1, 51, 'O', 'O'), (2, 51, 'C', 'O'), (2, 51, 'CA', 'O'), (43, 92, 'O', 'CB'), (43, 92, 'O', 'CD2')]

Prompt length: 95 tokens


In [17]:
#VALID_ATOMS

In [18]:
# Load model & tokenizer
import glob
import torch
from pathlib import Path
from transformers import LlamaForCausalLM
from experiments.exp4_contact_prediction.src.train import create_tokenizer, parse_generated_contacts

# Find latest checkpoint
ckpt_base = Path(CHECKPOINT_PATH)
ckpt_dirs = sorted(ckpt_base.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
if ckpt_dirs:
    ckpt_path = ckpt_dirs[-1]
else:
    ckpt_path = ckpt_base  # fallback: treat base as the checkpoint
print(f"Loading checkpoint: {ckpt_path}")

tokenizer = create_tokenizer()
model = LlamaForCausalLM.from_pretrained(str(ckpt_path), torch_dtype=torch.bfloat16)
model = model.to(DEVICE).eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

Loading checkpoint: ../../outputs/exp4/checkpoint-29500


Loading weights: 100%|██████████| 147/147 [00:00<00:00, 361.67it/s, Materializing param=model.norm.weight]                              


Model loaded: 984,475,648 parameters


In [ ]:
# Generate contacts: greedy + sampled rollouts for each prefix size
import time

end_token_id = tokenizer.convert_tokens_to_ids("<end>")
generated_contacts = {}  # prefix_size -> greedy contacts
rollout_contacts = {}    # prefix_size -> list of N_ROLLOUTS contact lists
timings = {}             # prefix_size -> {"greedy": seconds, "rollouts": [seconds, ...]}

def build_prompt(n_prefix):
    if n_prefix > 0 and gt_contacts:
        prefix_toks = []
        for p1, p2, a1, a2 in gt_contacts[:n_prefix]:
            prefix_toks.extend([f"<p{p1}>", f"<p{p2}>", f"<{a1}>", f"<{a2}>"])
        return base_prompt + " " + " ".join(prefix_toks)
    return base_prompt

def run_generation(prompt, do_sample=False):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=8192)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=end_token_id,
    )
    if do_sample:
        gen_kwargs.update(do_sample=True, temperature=1.0, top_k=0)
    else:
        gen_kwargs.update(do_sample=False)

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
    elapsed = time.time() - t0

    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=False)
    contacts, valid_grammar = parse_generated_contacts(gen_text.split())
    return contacts, valid_grammar, elapsed

for n_prefix in PREFIX_SIZES:
    prompt = build_prompt(n_prefix)
    timing_info = {}

    # Greedy
    contacts, valid, elapsed = run_generation(prompt, do_sample=False)
    if n_prefix > 0:
        contacts = list(gt_contacts[:n_prefix]) + list(contacts)
    generated_contacts[n_prefix] = contacts
    timing_info["greedy"] = elapsed
    print(f"Prefix {n_prefix:2d} greedy: {len(contacts):4d} contacts, valid={valid}, {elapsed:.1f}s")

    # Sampled rollouts
    rollouts = []
    rollout_times = []
    for r in range(N_ROLLOUTS):
        contacts_r, valid_r, elapsed_r = run_generation(prompt, do_sample=True)
        if n_prefix > 0:
            contacts_r = list(gt_contacts[:n_prefix]) + list(contacts_r)
        rollouts.append(contacts_r)
        rollout_times.append(elapsed_r)
    rollout_contacts[n_prefix] = rollouts
    timing_info["rollouts"] = rollout_times
    timings[n_prefix] = timing_info
    avg_t = np.mean(rollout_times)
    print(f"  rollouts: {N_ROLLOUTS}x, avg {avg_t:.1f}s each")

In [ ]:
# Build contact matrices (greedy + rollout frequency)
def contacts_to_matrix(contacts, seq_len):
    """Convert contact list to symmetric NxN binary matrix."""
    matrix = np.zeros((seq_len, seq_len), dtype=np.float32)
    for p1, p2, a1, a2 in contacts:
        if 1 <= p1 <= seq_len and 1 <= p2 <= seq_len:
            matrix[p1 - 1, p2 - 1] = 1
            matrix[p2 - 1, p1 - 1] = 1
    return matrix

def contacts_to_pair_set(contacts):
    return {(min(c[0], c[1]), max(c[0], c[1])) for c in contacts}

gt_matrix = contacts_to_matrix(gt_contacts, seq_len)
gt_pair_set = contacts_to_pair_set(gt_contacts)

# Greedy: separate prefix and generated-only matrices + accuracy
prefix_matrices = {}
gen_only_matrices = {}
gen_accuracy = {}

for n_prefix, contacts in generated_contacts.items():
    prefix_contacts = contacts[:n_prefix] if n_prefix > 0 else []
    gen_only_contacts = contacts[n_prefix:] if n_prefix > 0 else contacts

    prefix_matrices[n_prefix] = contacts_to_matrix(prefix_contacts, seq_len)
    gen_only_matrices[n_prefix] = contacts_to_matrix(gen_only_contacts, seq_len)

    gen_pairs = contacts_to_pair_set(gen_only_contacts)
    n_gen = len(gen_pairs)
    n_correct = len(gen_pairs & gt_pair_set)
    precision = n_correct / n_gen if n_gen > 0 else 0.0
    recall = n_correct / len(gt_pair_set) if gt_pair_set else 0.0
    gen_accuracy[n_prefix] = {
        "n_gen_pairs": n_gen, "n_correct": n_correct,
        "precision": precision, "recall": recall,
    }

# Rollout frequency matrices (fraction of rollouts containing each residue pair)
rollout_freq_matrices = {}
rollout_accuracy = {}

for n_prefix, rollouts in rollout_contacts.items():
    freq = np.zeros((seq_len, seq_len), dtype=np.float32)
    for contacts in rollouts:
        gen_only = contacts[n_prefix:] if n_prefix > 0 else contacts
        m = contacts_to_matrix(gen_only, seq_len)
        freq += m
    freq /= len(rollouts)
    rollout_freq_matrices[n_prefix] = freq

    # Accuracy: a pair is "predicted" if it appears in >50% of rollouts
    predicted_pairs = set()
    for i in range(seq_len):
        for j in range(i + 1, seq_len):
            if freq[i, j] > 0.5:
                predicted_pairs.add((i + 1, j + 1))
    n_pred = len(predicted_pairs)
    n_correct = len(predicted_pairs & gt_pair_set)
    precision = n_correct / n_pred if n_pred > 0 else 0.0
    recall = n_correct / len(gt_pair_set) if gt_pair_set else 0.0
    rollout_accuracy[n_prefix] = {
        "n_pred_pairs": n_pred, "n_correct": n_correct,
        "precision": precision, "recall": recall,
    }

print(f"Ground truth: {len(gt_pair_set)} unique residue pairs\n")
print("Greedy:")
for k, acc in gen_accuracy.items():
    print(f"  Prefix {k:2d}: {acc['n_gen_pairs']} gen pairs, "
          f"{acc['n_correct']} correct, "
          f"prec={acc['precision']:.1%}, rec={acc['recall']:.1%}")
print(f"\nRollout consensus (>50% of {N_ROLLOUTS} rollouts):")
for k, acc in rollout_accuracy.items():
    print(f"  Prefix {k:2d}: {acc['n_pred_pairs']} pred pairs, "
          f"{acc['n_correct']} correct, "
          f"prec={acc['precision']:.1%}, rec={acc['recall']:.1%}")

In [ ]:
# Plot greedy heatmaps with red circles on prefix contacts
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Circle

n_cols = 1 + len(PREFIX_SIZES)
fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4.8), squeeze=False)
axes = axes[0]

cmap_gt = ListedColormap(["white", "black"])

# Ground truth
axes[0].imshow(gt_matrix, cmap=cmap_gt, vmin=0, vmax=1, origin="upper", aspect="equal")
axes[0].set_title(f"Ground Truth\n{len(gt_pair_set)} residue pairs")
axes[0].set_xlabel("Residue")
axes[0].set_ylabel("Residue")

# Generated panels
for i, n_prefix in enumerate(PREFIX_SIZES):
    ax = axes[i + 1]
    acc = gen_accuracy[n_prefix]

    # Show generated contacts in grey
    ax.imshow(gen_only_matrices[n_prefix], cmap=cmap_gt, vmin=0, vmax=1,
              origin="upper", aspect="equal")

    # Draw red circles around prefix contacts
    if n_prefix > 0:
        prefix_pairs_plotted = set()
        for p1, p2, a1, a2 in gt_contacts[:n_prefix]:
            pair = (min(p1, p2) - 1, max(p1, p2) - 1)
            if pair not in prefix_pairs_plotted:
                prefix_pairs_plotted.add(pair)
                for (r, c) in [(pair[0], pair[1]), (pair[1], pair[0])]:
                    circle = Circle((c, r), radius=1.5, fill=False,
                                    edgecolor="red", linewidth=1.5)
                    ax.add_patch(circle)

    ax.set_title(f"Greedy, prefix={n_prefix}")
    ax.set_xlabel("Residue")

    stats_text = (
        f"Prec: {acc['precision']:.0%}  Rec: {acc['recall']:.0%}\n"
        f"{acc['n_correct']}/{acc['n_gen_pairs']} correct"
    )
    ax.text(0.5, -0.02, stats_text, transform=ax.transAxes, ha="center", va="top",
            fontsize=8, fontfamily="monospace")

legend_elements = [
    Patch(facecolor="black", label="Generated"),
    Patch(facecolor="white", edgecolor="red", linewidth=1.5, label="Prefix (ground truth)"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle(
    f"Greedy Contact Maps: {PDB_ID} ({seq_len} residues, cutoff={CONTACT_DISTANCE_CUTOFF}\u00c5)",
    fontsize=14, y=1.02,
)
plt.tight_layout()
plt.subplots_adjust(bottom=0.18)
plt.show()

In [ ]:
# Plot rollout frequency heatmaps (soft contact maps)
n_cols = 1 + len(PREFIX_SIZES)
fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 5.2), squeeze=False)
axes = axes[0]

# Ground truth (binary) for reference
axes[0].imshow(gt_matrix, cmap="Greys", vmin=0, vmax=1, origin="upper", aspect="equal")
axes[0].set_title(f"Ground Truth\n{len(gt_pair_set)} residue pairs")
axes[0].set_xlabel("Residue")
axes[0].set_ylabel("Residue")

for i, n_prefix in enumerate(PREFIX_SIZES):
    ax = axes[i + 1]
    acc = rollout_accuracy[n_prefix]
    freq = rollout_freq_matrices[n_prefix]

    im = ax.imshow(freq, cmap="hot_r", vmin=0, vmax=1, origin="upper", aspect="equal")

    # Draw red circles around prefix contacts
    if n_prefix > 0:
        prefix_pairs_plotted = set()
        for p1, p2, a1, a2 in gt_contacts[:n_prefix]:
            pair = (min(p1, p2) - 1, max(p1, p2) - 1)
            if pair not in prefix_pairs_plotted:
                prefix_pairs_plotted.add(pair)
                for (r, c) in [(pair[0], pair[1]), (pair[1], pair[0])]:
                    circle = Circle((c, r), radius=1.5, fill=False,
                                    edgecolor="blue", linewidth=1.5)
                    ax.add_patch(circle)

    ax.set_title(f"Rollouts, prefix={n_prefix}")
    ax.set_xlabel("Residue")

    stats_text = (
        f">50% consensus: prec={acc['precision']:.0%} rec={acc['recall']:.0%}\n"
        f"{acc['n_correct']}/{acc['n_pred_pairs']} correct"
    )
    ax.text(0.5, -0.02, stats_text, transform=ax.transAxes, ha="center", va="top",
            fontsize=8, fontfamily="monospace")

# Colorbar
cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.8, pad=0.02)
cbar.set_label(f"Fraction of {N_ROLLOUTS} rollouts", fontsize=9)

legend_elements = [
    Patch(facecolor="white", edgecolor="blue", linewidth=1.5, label="Prefix (ground truth)"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=1, fontsize=9,
           bbox_to_anchor=(0.45, -0.02))

fig.suptitle(
    f"Rollout Frequency Maps: {PDB_ID} ({seq_len} residues, {N_ROLLOUTS} rollouts)",
    fontsize=14, y=1.02,
)
plt.tight_layout()
plt.subplots_adjust(bottom=0.16)
plt.show()

In [ ]:
# Plot generation timings
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(PREFIX_SIZES))
width = 0.25

# Greedy times
greedy_times = [timings[p]["greedy"] for p in PREFIX_SIZES]
bars1 = ax.bar(x - width/2, greedy_times, width, label="Greedy", color="#333333")

# Rollout times (mean + std)
rollout_means = [np.mean(timings[p]["rollouts"]) for p in PREFIX_SIZES]
rollout_stds = [np.std(timings[p]["rollouts"]) for p in PREFIX_SIZES]
bars2 = ax.bar(x + width/2, rollout_means, width, yerr=rollout_stds,
               label=f"Rollout (mean±std, N={N_ROLLOUTS})", color="#E57373", capsize=3)

ax.set_xlabel("Prefix size")
ax.set_ylabel("Time (seconds)")
ax.set_title(f"Generation Time: {PDB_ID} ({seq_len} residues)")
ax.set_xticks(x)
ax.set_xticklabels([str(p) for p in PREFIX_SIZES])
ax.legend()

# Annotate bars with values
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}s", ha="center", va="bottom", fontsize=8)
for bar, std in zip(bars2, rollout_stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.3,
            f"{bar.get_height():.1f}s", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()
